# Coleta de reviews do IMDb

Notebook para baixar reviews de qualquer filme do IMDb e montar um dataset pronto
para analise de sentimento.

**Como funciona.** O `www.imdb.com` bloqueia scraping tradicional: responde `202` com
corpo vazio para requisicoes automatizadas, e o antigo endpoint `/reviews/_ajax` foi
descontinuado. Entao aqui usamos o mesmo endpoint GraphQL que o proprio site consome
(`caching.graphql.imdb.com`), paginando por cursor.

**Dependencias.** Apenas a stdlib do Python + `pandas`. Nada de `requests`,
`selenium` ou `beautifulsoup`.

**Uso.** Edite a celula de configuracao abaixo e rode tudo (`Run All`).

## 1. Configuracao

Unica celula que voce precisa mexer para trocar de filme.

In [1]:
# Filme alvo: informe o NOME (a busca resolve o ID) ou o IMDB_ID direto.
MOVIE_TITLE = "Spider-Man: Brand New Day"
IMDB_ID = "tt1114677"  # ex: "tt22084616" — se preenchido, ignora MOVIE_TITLE

# Quantos reviews baixar
N_REVIEWS = 1000

# Ordenacao: HELPFULNESS_SCORE | SUBMISSION_DATE | USER_RATING
#            | TOTAL_VOTES | SUBMITTER_REVIEW_COUNT
SORT_BY = "HELPFULNESS_SCORE"
SORT_ORDER = "DESC"  # DESC ou ASC

# Educacao / boa vizinhanca: pausa entre requisicoes (segundos)
DELAY = 0.5

# Prefixo dos arquivos de saida (.csv e .json)
OUTPUT_PREFIX = None  # None = gera automaticamente a partir do ID

## 2. Imports e helper de rede

`_http_json` centraliza os headers que o endpoint espera, trata gzip e faz retry com
backoff exponencial (util quando o IMDb devolve 429/503).

In [2]:
import csv
import gzip
import json
import ssl
import sys
import time
import urllib.error
import urllib.parse
import urllib.request

import pandas as pd

GRAPHQL_URL = "https://caching.graphql.imdb.com/"
SUGGESTION_URL = "https://v3.sg.media-imdb.com/suggestion/x/{}.json"

# O Python do python.org no macOS nao usa o keychain do sistema: sem isto todo
# urlopen morre com CERTIFICATE_VERIFY_FAILED. O certifi resolve na hora.
try:
    import certifi

    CA_BUNDLE = certifi.where()
    SSL_CONTEXT = ssl.create_default_context(cafile=CA_BUNDLE)
except ModuleNotFoundError:
    CA_BUNDLE = "padrao do sistema (certifi nao instalado)"
    SSL_CONTEXT = ssl.create_default_context()

CERT_HELP = (
    "Falha ao validar o certificado SSL do IMDb. Duas saidas:\n"
    "  1) instale o certifi e rode esta celula de novo:  %pip install certifi\n"
    "  2) ou rode uma vez o instalador de certificados do seu Python:\n"
    "     open '/Applications/Python 3.10/Install Certificates.command'"
)

# Conjunto minimo, verificado por teste (3 rodadas por caso, com ids de filme
# diferentes para nao pegar cache do endpoint):
#   Content-Type       -> obrigatorio: sem ele o GraphQL responde 415
#   Referer do imdb.com-> sozinho ja libera (de outro dominio da 403)
#   x-imdb-client-name -> sozinho tambem libera; o valor nao e validado
# Os dois ultimos sao redundantes entre si: qualquer um basta. Mantidos ambos
# de proposito, para o caso de o IMDb apertar um dos caminhos.
# Removidos por serem inertes: User-Agent, Origin, Accept, Accept-Language
# (UA de navegador sozinho da 403; UA de curl + client-name da 200).
HEADERS = {
    "Content-Type": "application/json",
    "Referer": "https://www.imdb.com/",
    "x-imdb-client-name": "imdb-web-next",
    "Accept-Encoding": "gzip",  # se a resposta vier comprimida, _http_json descomprime
}


def _http_json(url, payload=None, retries=5):
    """GET (payload=None) ou POST JSON, com retry e backoff exponencial."""
    data = json.dumps(payload).encode("utf-8") if payload is not None else None

    for attempt in range(retries):
        try:
            req = urllib.request.Request(url, data=data, headers=HEADERS)
            with urllib.request.urlopen(req, timeout=30, context=SSL_CONTEXT) as resp:
                raw = resp.read()
                if resp.headers.get("Content-Encoding") == "gzip":
                    raw = gzip.decompress(raw)
                return json.loads(raw.decode("utf-8"))

        except urllib.error.HTTPError as exc:
            # 4xx que nao seja rate-limit nao melhora com retry
            if exc.code not in (429, 500, 502, 503, 504) or attempt == retries - 1:
                raise
        except (urllib.error.URLError, TimeoutError) as exc:
            # certificado quebrado tambem nao melhora com retry: falha na hora
            if isinstance(getattr(exc, "reason", None), ssl.SSLCertVerificationError):
                raise RuntimeError(CERT_HELP) from exc
            if attempt == retries - 1:
                raise

        wait = 2 ** attempt
        print(f"  ! tentativa {attempt + 1}/{retries} falhou; aguardando {wait}s")
        time.sleep(wait)


print(f"helpers carregados | Python {'.'.join(map(str, sys.version_info[:3]))}")
print(f"certificados: {CA_BUNDLE}")

helpers carregados | Python 3.10.11
certificados: /Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/certifi/cacert.pem


## 3. Descobrir o ID do filme

A API de sugestoes do IMDb (a mesma da barra de busca do site) resolve nome -> ID.

> **Confira o resultado.** O ranking e por popularidade atual, nao por match exato.
> `"Dune"` devolve *Dune: Part Two* e `"Avatar"` devolve *Avatar Aang*, nao os filmes
> que voce provavelmente quer. A celula imprime os outros candidatos: se o escolhido
> estiver errado, cole o ID certo em `IMDB_ID` na celula 1 e rode de novo.

In [3]:
def search_titles(query, limit=6):
    """Lista os candidatos do IMDb para um nome, em ordem de relevancia."""
    payload = _http_json(SUGGESTION_URL.format(urllib.parse.quote(query)))
    return [
        {"id": it["id"], "name": it.get("l"), "type": it.get("q"), "year": it.get("y")}
        for it in payload.get("d", [])
        # entradas editoriais (listas, artigos) nao tem id comecando com "tt"
        if str(it.get("id", "")).startswith("tt")
    ][:limit]


def find_title_id(query, limit=6):
    """Resolve nome -> (id, titulo, ano), mostrando os demais candidatos.

    A busca do IMDb rankeia por popularidade atual, nao por match exato:
    "Dune" devolve Dune: Part Two (2024) e "Avatar" devolve Avatar Aang (2026),
    nao os filmes que voce provavelmente quer. Por isso os outros candidatos
    aparecem — se o escolhido estiver errado, copie o ID certo para IMDB_ID.
    """
    candidatos = search_titles(query, limit)
    if not candidatos:
        raise ValueError(f"nenhum titulo encontrado para {query!r}")

    escolhido, *outros = candidatos
    print(f"'{query}' -> {escolhido['id']} | {escolhido['name']} "
          f"({escolhido['year']}) [{escolhido['type']}]")

    if outros:
        print("\n  outros candidatos — se o de cima estiver errado, cole o ID em IMDB_ID:")
        for c in outros:
            print(f"    {c['id']}  {c['name']} ({c['year']}) [{c['type']}]")

    return escolhido["id"], escolhido["name"], escolhido["year"]


if IMDB_ID:
    title_id = IMDB_ID
    print(f"usando ID fixo: {title_id}")
else:
    title_id, resolved_name, resolved_year = find_title_id(MOVIE_TITLE)

usando ID fixo: tt1114677


## 4. A query GraphQL

Cada `node` e um review. O `pageInfo.endCursor` alimenta a proxima pagina.

> O endpoint rejeita `first > 50` com um `BAD_USER_INPUT` generico — por isso o
> `PAGE_SIZE` fixo em 50.

In [4]:
PAGE_SIZE = 50  # maximo aceito pelo endpoint

REVIEWS_QUERY = """
query TitleReviews($id: ID!, $first: Int!, $after: ID, $sort: ReviewsSortBy!, $order: SortOrder!) {
  title(id: $id) {
    titleText { text }
    releaseYear { year }
    ratingsSummary { aggregateRating voteCount }
    reviews(first: $first, after: $after, sort: { by: $sort, order: $order }) {
      total
      pageInfo { hasNextPage endCursor }
      edges {
        node {
          id
          summary { originalText }
          text { originalText { plainText } }
          authorRating
          submissionDate
          spoiler
          helpfulness { upVotes downVotes }
          author { nickName userId }
        }
      }
    }
  }
}
"""

print("query definida")

query definida


## 5. Loop de paginacao

Percorre o cursor ate juntar `N_REVIEWS` (ou ate acabarem os reviews do filme).
Deduplica por `review_id` e deriva um rotulo de sentimento a partir da nota do autor.

In [5]:
def sentiment_from_rating(rating):
    """Rotulo derivado da nota do autor (1-10), para usar como label de treino."""
    if rating is None:
        return "unknown"
    if rating <= 4:
        return "negative"
    if rating <= 6:
        return "neutral"
    return "positive"


def fetch_reviews(title_id, limit, sort=SORT_BY, order=SORT_ORDER, delay=DELAY):
    reviews, seen, cursor, meta, page = [], set(), None, {}, 0

    while len(reviews) < limit:
        page += 1
        variables = {
            "id": title_id,
            "first": min(PAGE_SIZE, limit - len(reviews)),
            "sort": sort,
            "order": order,
        }
        if cursor:
            variables["after"] = cursor

        payload = _http_json(GRAPHQL_URL, {"query": REVIEWS_QUERY, "variables": variables})

        if "errors" in payload:
            raise RuntimeError(json.dumps(payload["errors"], indent=2))

        title = (payload.get("data") or {}).get("title")
        if not title:
            raise RuntimeError(f"titulo {title_id} nao encontrado")

        block = title["reviews"]

        if not meta:
            meta = {
                "imdb_id": title_id,
                "title": title["titleText"]["text"],
                "year": (title.get("releaseYear") or {}).get("year"),
                "imdb_rating": (title.get("ratingsSummary") or {}).get("aggregateRating"),
                "imdb_votes": (title.get("ratingsSummary") or {}).get("voteCount"),
                "reviews_available": block["total"],
            }
            print(f"{meta['title']} ({meta['year']}) — nota {meta['imdb_rating']} — "
                  f"{meta['reviews_available']} reviews disponiveis\n")

        for edge in block["edges"]:
            node = edge["node"]
            if node["id"] in seen:
                continue
            seen.add(node["id"])

            rating = node.get("authorRating")
            reviews.append({
                "review_id": node["id"],
                "title": (node.get("summary") or {}).get("originalText") or "",
                "review": ((node.get("text") or {}).get("originalText") or {}).get("plainText") or "",
                "rating": rating,
                "sentiment": sentiment_from_rating(rating),
                "date": node.get("submissionDate"),
                "author": (node.get("author") or {}).get("nickName"),
                "author_id": (node.get("author") or {}).get("userId"),
                "spoiler": node.get("spoiler"),
                "helpful_up": (node.get("helpfulness") or {}).get("upVotes"),
                "helpful_down": (node.get("helpfulness") or {}).get("downVotes"),
                "url": f"https://www.imdb.com/review/{node['id']}/",
            })

        pct = 100 * len(reviews) / limit
        print(f"\rpagina {page:>3} | {len(reviews):>5}/{limit} reviews ({pct:5.1f}%)", end="")

        if not block["pageInfo"]["hasNextPage"]:
            print("\n\nacabaram os reviews disponiveis antes de atingir a meta")
            break

        cursor = block["pageInfo"]["endCursor"]
        time.sleep(delay)

    print(f"\n\nconcluido: {len(reviews)} reviews")
    return reviews[:limit], meta


print("funcao pronta")

funcao pronta


## 6. Rodar a coleta

Com os defaults (1000 reviews, `DELAY = 0.5`), leva cerca de 1 minuto.

In [6]:
reviews, meta = fetch_reviews(title_id, N_REVIEWS)

Hannah Montana: The Movie (2009) — nota 4.8 — 183 reviews disponiveis

pagina   4 |   183/1000 reviews ( 18.3%)

acabaram os reviews disponiveis antes de atingir a meta


concluido: 183 reviews


## 7. DataFrame

`df` e o dataset final, pronto para o pipeline de NLP.

In [7]:
df = pd.DataFrame(reviews)
df["date"] = pd.to_datetime(df["date"])
df["rating"] = pd.to_numeric(df["rating"])
df["n_chars"] = df["review"].str.len()
df["n_words"] = df["review"].str.split().str.len()

print(f"shape: {df.shape}")
df.head()

shape: (183, 14)


,review_id,title,review,rating,sentiment,date,author,author_id,spoiler,helpful_up,helpful_down,url,n_chars,n_words
0,rw2055940,"It wasn't that bad, people.",I just wanted to comment on how completely rid...,6.0,neutral,2009-04-22,weldermommy,ur16162521,False,166,85,https://www.imdb.com/review/rw2055940/,1044,195
1,rw3026375,"Decent, but ambiguous",It's hard to believe that it haven't been a lo...,5.0,neutral,2014-06-01,Tinny-Tinette,ur27797700,True,5,0,https://www.imdb.com/review/rw3026375/,3009,514
2,rw2113716,"An OK film I guess, a good conclusion to the s...","With my laptop with me, my sister and me decid...",5.0,neutral,2009-08-19,karl_with-12,ur21671556,True,19,11,https://www.imdb.com/review/rw2113716/,3785,700
3,rw2052155,geez people chill out!,Some of these people remind me of freaking' cr...,5.0,neutral,2009-04-13,jkae89,ur20138134,False,42,39,https://www.imdb.com/review/rw2052155/,591,99
4,rw2211211,"Hannah Montana ""The Movie"": As Silly as They Come",I cannot imagine how anyone in their right min...,3.0,negative,2010-02-22,jonathanruano,ur20682974,True,12,7,https://www.imdb.com/review/rw2211211/,2438,431


## 8. Sanidade e distribuicao

Vale conferir antes de treinar qualquer coisa: duplicatas, textos vazios e
o balanceamento das classes.

In [8]:
print(f"linhas ................ {len(df)}")
print(f"review_ids unicos ..... {df['review_id'].nunique()}")
print(f"textos vazios ......... {(df['review'].str.strip() == '').sum()}")
print(f"sem nota do autor ..... {df['rating'].isna().sum()}")
print(f"periodo ............... {df['date'].min():%Y-%m-%d} a {df['date'].max():%Y-%m-%d}")
print(f"tamanho medio ......... {df['n_words'].mean():.0f} palavras "
      f"(mediana {df['n_words'].median():.0f})")
print(f"nota media do autor ... {df['rating'].mean():.2f}")

print("\ndistribuicao de sentimento:")
dist = df["sentiment"].value_counts()
for label, count in dist.items():
    bar = "#" * round(50 * count / len(df))
    print(f"  {label:<9} {count:>5} ({100 * count / len(df):4.1f}%) {bar}")

print("\ndistribuicao de notas:")
for rating, count in df["rating"].value_counts().sort_index().items():
    bar = "#" * round(50 * count / len(df))
    print(f"  {int(rating):>2}/10 {count:>5} {bar}")

linhas ................ 183
review_ids unicos ..... 183
textos vazios ......... 0
sem nota do autor ..... 7
periodo ............... 2009-04-07 a 2026-03-14
tamanho medio ......... 225 palavras (mediana 169)
nota media do autor ... 6.76

distribuicao de sentimento:
  positive    105 (57.4%) #############################
  negative     36 (19.7%) ##########
  neutral      35 (19.1%) ##########
  unknown       7 ( 3.8%) ##

distribuicao de notas:
   1/10    19 #####
   2/10     8 ##
   3/10     4 #
   4/10     5 #
   5/10    19 #####
   6/10    16 ####
   7/10    17 #####
   8/10    26 #######
   9/10    15 ####
  10/10    47 #############


## 9. Exportar

Gera `.csv` (para o pipeline) e `.json` (com os metadados do filme junto).

In [9]:
prefix = OUTPUT_PREFIX or f"imdb_{title_id}_reviews"

df.drop(columns=["n_chars", "n_words"]).to_csv(
    f"{prefix}.csv", index=False, quoting=csv.QUOTE_ALL, encoding="utf-8"
)

with open(f"{prefix}.json", "w", encoding="utf-8") as fh:
    json.dump({"meta": meta, "count": len(reviews), "reviews": reviews},
              fh, ensure_ascii=False, indent=2)

print(f"salvo: {prefix}.csv")
print(f"salvo: {prefix}.json")

salvo: imdb_tt1114677_reviews.csv
salvo: imdb_tt1114677_reviews.json


## 10. Recarregar depois

Para nao rodar o scraping de novo em outra sessao:

```python
df = pd.read_csv("imdb_tt22084616_reviews.csv")
```

## Trocar de filme

Volte na celula 1, mude `MOVIE_TITLE` e rode tudo de novo. Exemplos:

| Filme | `MOVIE_TITLE` |
| --- | --- |
| Spider-Man: Brand New Day | `"Spider-Man: Brand New Day"` |
| Oppenheimer | `"Oppenheimer"` |
| Interstellar | `"Interstellar"` |

Para montar um dataset com varios filmes:

```python
frames = []
for nome in ["Oppenheimer", "Barbie", "Interstellar"]:
    tid, *_ = find_title_id(nome)
    revs, m = fetch_reviews(tid, 500)
    frames.append(pd.DataFrame(revs).assign(movie=m["title"]))

todos = pd.concat(frames, ignore_index=True)
```

## Nota sobre uso dos dados

O IMDb declara nas respostas da API que o uso e restrito a fins nao comerciais.
Isso cobre trabalho academico, mas nao a redistribuicao publica do dataset.